# NullVector — End-to-End LangGraph QA Agent with PostgreSQL

Builds a **ReAct agent** that answers questions about a previously ingested document
stored in PostgreSQL. The agent uses NullVector's retrieval stack as its tool layer.

```
User question
 └─ LangGraph ReAct loop
     ├─ search_document(query)  → NullVector QueryPlanner + RetrievalRanker
     └─ get_page_text(page)     → Full page text from corpus
 └─ Grounded answer with page citations
```

**Prerequisites:**
- Run `03_nullvector_ingestion_postgres.ipynb` first to populate the corpus
- `/tmp/nullvector_manifest_refs.json` must exist (written by the ingestion notebook)
- `OPENROUTER_API_KEY` env var set
- `uv add langchain-openai langgraph langchain-core` (if not already installed)

In [ ]:
import json
import os
from pathlib import Path
from typing import Annotated, TypedDict

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

from nullvector.domain.retrieval import RetrievalUnitType
from nullvector.retrieval import QueryPlanner, RetrievalRanker, RetrievalService
from nullvector.retrieval.load import load_retrieval_corpus
from nullvector.storage.config import PostgresStorageConfig

## Configuration

In [ ]:
POSTGRES_URI = "postgresql://REDACTED_DB_CRED@localhost:5432/app"
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")

storage = PostgresStorageConfig(conninfo=POSTGRES_URI)

# Load manifest refs written by the ingestion notebook
refs_path = Path("/tmp/nullvector_manifest_refs.json")
refs = json.loads(refs_path.read_text())
CORPUS_PATH = refs["corpus_path"]

print(f"Document ID  : {refs.get('document_id', 'n/a')}")
print(f"Corpus path  : {CORPUS_PATH}")
print(f"Total units  : {refs.get('total_corpus_units', 'n/a')}")

## Load Retrieval Corpus and Services

`load_retrieval_corpus` deserialises the `RetrievalCorpus` from a PostgreSQL artifact
ref or filesystem path. `RetrievalService` wraps the `QueryPlanner` (deterministic
intent detection) and `RetrievalRanker` (BM25-style scoring with modality awareness).

In [ ]:
corpus = load_retrieval_corpus(CORPUS_PATH, storage=storage)

planner = QueryPlanner()
ranker = RetrievalRanker()
service = RetrievalService(planner, ranker)

print(f"Loaded {len(corpus.units)} units  |  document_id={corpus.document_id}")

from collections import Counter

for utype, count in sorted(Counter(u.unit_type for u in corpus.units).items()):
    print(f"  {utype:<25}: {count}")

## Define Agent Tools

Two tools expose NullVector retrieval to the LangGraph agent:

- **`search_document`** — runs `RetrievalService.search()` with query planning and
  ranking; returns top-k excerpts with page numbers, unit type, and score.
- **`get_page_text`** — returns the full raw text of a specific page (0-indexed).

In [ ]:
@tool
def search_document(query: str, limit: int = 5) -> str:
    """Search the document corpus for content relevant to the query.

    Returns ranked excerpts with page numbers, unit types, and relevance scores.
    Use this before answering any question about the document.
    """
    hits = service.search(corpus=corpus, query=query, limit=limit)
    if not hits:
        return "No results found for this query."
    parts = []
    for i, hit in enumerate(hits, 1):
        u = hit.unit
        page = u.page_span.start_page if u.page_span else "?"
        excerpt = (u.text or "")[:350].replace("\n", " ")
        title = u.metadata.get("title", "") if u.metadata else ""
        header = f"[{i}] page={page}  type={u.unit_type}  score={hit.score:.3f}"
        if title:
            header += f"  title='{title}'"
        parts.append(f"{header}\n{excerpt}")
    return "\n\n".join(parts)


@tool
def get_page_text(page_number: int) -> str:
    """Get the complete text of a specific page (0-indexed).

    Use when you need the full context of a page rather than a ranked excerpt.
    """
    for unit in corpus.units:
        if (
            unit.unit_type == RetrievalUnitType.PAGE_TEXT
            and unit.page_span is not None
            and unit.page_span.start_page == page_number
        ):
            return unit.text or f"Page {page_number} has no text content."
    return f"Page {page_number} not found in the corpus."


TOOLS = [search_document, get_page_text]
TOOL_NODE = ToolNode(TOOLS)
print("Tools registered:", [t.name for t in TOOLS])

## Initialise the LLM

We use `ChatOpenAI` pointed at OpenRouter so any model can be swapped without code
changes. `bind_tools` attaches the tool schemas so the model can call them.

In [ ]:
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
).bind_tools(TOOLS)

print("LLM ready")

## Build the LangGraph ReAct Agent

The graph is a standard ReAct loop:

```
START → agent ──(tool_calls?)──► tools → agent (loop)
                └──(no calls)──► END
```

`MemorySaver` provides in-process thread-scoped memory for multi-turn conversations.

In [ ]:
SYSTEM_PROMPT = """You are a precise document QA assistant powered by NullVector.

Guidelines:
- Always call search_document before answering factual questions.
- Use get_page_text when you need the full context of a page.
- Cite the page number(s) from your search results in every answer.
- If the document does not contain information to answer the question, say so clearly.
- Keep answers concise and grounded in the retrieved text."""


class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


def call_llm(state: AgentState) -> dict[str, list[BaseMessage]]:
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    return {"messages": [llm.invoke(messages)]}


def should_continue(state: AgentState) -> str:
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return END


builder = StateGraph(AgentState)
builder.add_node("agent", call_llm)
builder.add_node("tools", TOOL_NODE)
builder.set_entry_point("agent")
builder.add_conditional_edges(
    "agent",
    should_continue,
    {"tools": "tools", END: END},
)
builder.add_edge("tools", "agent")

memory = MemorySaver()
app = builder.compile(checkpointer=memory)
print("Agent graph compiled")

## Visualise the Graph

In [ ]:
from IPython.display import Image

Image(app.get_graph().draw_mermaid_png())

## Helper: `ask()`

A thin wrapper around `app.invoke()` that keeps a conversation thread alive.
Each `thread_id` maintains its own message history via `MemorySaver`.

In [ ]:
def ask(question: str, *, thread_id: str = "main", verbose: bool = False) -> str:
    """Submit a question to the agent and return the final answer."""
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke({"messages": [HumanMessage(content=question)]}, config)
    messages = result["messages"]
    if verbose:
        for msg in messages:
            role = type(msg).__name__
            content = getattr(msg, "content", "") or ""
            tool_calls = getattr(msg, "tool_calls", [])
            if tool_calls:
                print(f"[{role}] → tool_calls: {[tc['name'] for tc in tool_calls]}")
            elif content:
                print(f"[{role}] {content[:120]}")
    last = messages[-1]
    return last.content if isinstance(last, AIMessage) else str(last)


print("ask() helper ready")

## Demo: Single-Turn Queries

In [ ]:
answer = ask("What is the main topic of this document?", thread_id="demo-1")
print(answer)

In [ ]:
answer = ask("List the main sections or chapters.", thread_id="demo-2")
print(answer)

In [ ]:
# Visual query — triggers VISUAL/UNRESOLVED_VISUAL unit routing in the planner
answer = ask("Are there any diagrams, charts, or figures? Describe what they show.", thread_id="demo-3")
print(answer)

## Demo: Multi-Turn Conversation

Using the same `thread_id` across calls preserves conversation context.
The agent can refer back to earlier answers.

In [ ]:
THREAD = "multi-turn-demo"

q1 = ask("What is the main topic of this document?", thread_id=THREAD)
print("Q1:", q1)
print()

In [ ]:
q2 = ask("Can you expand on the first section?", thread_id=THREAD)
print("Q2:", q2)
print()

In [ ]:
q3 = ask("What evidence does the document provide for that?", thread_id=THREAD)
print("Q3:", q3)

## Demo: Streaming

`app.stream()` with `stream_mode="values"` yields one event per state update.
We print only the final AIMessage (no tool calls) to get a streaming feel.

In [ ]:
config = {"configurable": {"thread_id": "stream-demo"}}
question = "Summarize the key findings or conclusions in this document."

print(f"Q: {question}\n")
for event in app.stream(
    {"messages": [HumanMessage(content=question)]},
    config,
    stream_mode="values",
):
    last = event["messages"][-1]
    if isinstance(last, AIMessage) and not last.tool_calls and last.content:
        print(last.content)

## Inspect Conversation State

`app.get_state()` returns the full message history for a thread.
Useful for debugging or auditing the agent's reasoning.

In [ ]:
state = app.get_state({"configurable": {"thread_id": THREAD}})
messages = state.values.get("messages", [])
print(f"Thread '{THREAD}' has {len(messages)} messages:\n")
for msg in messages:
    role = type(msg).__name__
    content = (getattr(msg, "content", "") or "")[:120]
    tool_calls = getattr(msg, "tool_calls", [])
    if tool_calls:
        print(f"  [{role}] tool_calls={[tc['name'] for tc in tool_calls]}")
    elif content:
        print(f"  [{role}] {content}")

## Direct Retrieval (Without Agent)

You can also call `RetrievalService.search()` directly to inspect what
the agent sees before the LLM processes it.

In [ ]:
hits = service.search(corpus=corpus, query="key findings conclusions", limit=5)
print(f"Direct retrieval: {len(hits)} hits\n")
for i, hit in enumerate(hits, 1):
    u = hit.unit
    page = u.page_span.start_page if u.page_span else "?"
    excerpt = (u.text or "")[:200].replace("\n", " ")
    print(f"[{i}] score={hit.score:.3f}  page={page}  type={u.unit_type}")
    print(f"     {excerpt}")
    print()